# Circuit Analysis: OV Circuits, QK Circuits, and Composition

Transformers implement **circuits** — compositions of attention heads that perform specific computations. Understanding these circuits is central to mechanistic interpretability.

This notebook covers:
1. **OV circuits**: What information does a head write? (attended-to token -> residual stream)
2. **QK circuits**: What does a head attend to? (query token -> key token)
3. **Full circuits**: Token-to-token maps through individual heads
4. **Direct logit attribution**: How much does each head affect the output?
5. **Composition scores**: How strongly do heads interact across layers?
6. **Weight processing**: Folding LayerNorm for cleaner analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk.circuits import (
    ov_circuit, qk_circuit, full_ov_circuit, full_qk_circuit,
    direct_logit_attribution, residual_stream_attribution,
    all_composition_scores,
    prev_token_score, induction_score,
)
from irtk.weight_processing import fold_layer_norm, center_unembed, center_writing_weights
from irtk import vis

print("Loading GPT-2 Small...")
model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Loaded: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")

## 1. OV Circuits: What a Head Writes

The **OV circuit** of a head is `W_V @ W_O`. It determines: when a head attends to a source token, what information does it copy into the residual stream?

The **full OV circuit** `W_E @ W_V @ W_O @ W_U` maps: "if I attend to token X, how does that change the logits?"

This is especially useful for identifying **copying heads** (where attending to token X promotes token X in the output).

In [ ]:
# Let's look at the full OV circuit of a head known to be important in GPT-2
# Head L9H9 is a known name mover head
layer, head = 9, 9

fov = full_ov_circuit(model, layer, head)
print(f"Full OV circuit shape: {fov.shape}")
print(f"(This is a {fov.shape[0]} x {fov.shape[1]} matrix: input token -> output token)")

# Get the diagonal: how much does attending to token X promote token X?
ov_mat = np.array(fov.AB)
diagonal = np.diag(ov_mat)

# A positive diagonal means the head copies tokens (attending to X promotes X)
print(f"\nMean diagonal (copy score): {diagonal.mean():.4f}")
print(f"Mean off-diagonal: {(ov_mat.sum() - diagonal.sum()) / (ov_mat.size - len(diagonal)):.4f}")

# Compare with a random head
fov_random = full_ov_circuit(model, 0, 0)
diag_random = np.diag(np.array(fov_random.AB))
print(f"\nL0H0 mean diagonal (for comparison): {diag_random.mean():.4f}")

In [ ]:
# Scan all heads for copy behavior
copy_scores = np.zeros((model.cfg.n_layers, model.cfg.n_heads))

for l in range(model.cfg.n_layers):
    for h in range(model.cfg.n_heads):
        fov = full_ov_circuit(model, l, h)
        mat = np.array(fov.AB)
        copy_scores[l, h] = np.mean(np.diag(mat))

fig, ax = vis.plot_head_summary(
    copy_scores,
    title="OV Copy Score (diagonal of full OV circuit)",
    cmap="RdBu_r",
    vmin=-0.5, vmax=0.5,
)
plt.show()

# Top copying heads
print("Top 5 copy heads (positive = copies, negative = suppresses):")
flat = copy_scores.flatten()
for i in np.argsort(np.abs(flat))[::-1][:5]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: {copy_scores[l,h]:+.4f}")

## 2. Direct Logit Attribution

Each attention head writes to the residual stream. We can project each head's output through the unembedding to see its direct effect on the logits. This tells us which heads are most responsible for the model's prediction.

In [ ]:
prompt = "The Eiffel Tower is located in the city of"
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens)

# What does the model predict?
pred_token = int(jnp.argmax(logits[-1]))
pred_str = tokenizer.decode([pred_token])
print(f"Prompt: {prompt!r}")
print(f"Prediction: {pred_str!r} (token {pred_token})")

# Direct logit attribution for the predicted token
head_attrs = direct_logit_attribution(model, cache, token=pred_token)

fig, ax = vis.plot_head_summary(
    head_attrs,
    title=f"Direct Logit Attribution for {pred_str!r}",
    cmap="RdBu_r",
    vmin=-2, vmax=2,
)
plt.show()

print("Top 5 heads promoting this prediction:")
flat = head_attrs.flatten()
for i in np.argsort(flat)[::-1][:5]:
    l = i // model.cfg.n_heads
    h = i % model.cfg.n_heads
    print(f"  L{l}H{h}: {head_attrs[l,h]:+.3f}")

In [ ]:
# Full residual stream attribution: embed + each attn + each MLP
component_attrs = residual_stream_attribution(model, cache, token=pred_token)

names = list(component_attrs.keys())
values = [component_attrs[n] for n in names]

fig, ax = vis.plot_logit_attribution(
    np.array(values), names,
    title=f"Residual Stream Attribution for {pred_str!r}"
)
plt.show()

## 3. Composition Scores

Heads in later layers read from the residual stream, which includes outputs from earlier heads. **Composition** measures how strongly one head's output feeds into another head's computation.

- **QK composition**: Head A's output is used as a query/key in head B (affects what B attends to)
- **OV composition**: Head A's output is used as a value in head B (affects what B copies)

In [ ]:
# Compute QK composition scores between all pairs of heads
print("Computing composition scores...")
qk_scores = all_composition_scores(model, "qk")
ov_scores = all_composition_scores(model, "ov")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

im1 = ax1.imshow(qk_scores, cmap="viridis", aspect="auto")
ax1.set_title("QK Composition Scores")
ax1.set_xlabel("Destination Head")
ax1.set_ylabel("Source Head")
for i in range(1, model.cfg.n_layers):
    pos = i * model.cfg.n_heads - 0.5
    ax1.axhline(y=pos, color='white', linewidth=0.3, alpha=0.5)
    ax1.axvline(x=pos, color='white', linewidth=0.3, alpha=0.5)
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(ov_scores, cmap="viridis", aspect="auto")
ax2.set_title("OV Composition Scores")
ax2.set_xlabel("Destination Head")
ax2.set_ylabel("Source Head")
for i in range(1, model.cfg.n_layers):
    pos = i * model.cfg.n_heads - 0.5
    ax2.axhline(y=pos, color='white', linewidth=0.3, alpha=0.5)
    ax2.axvline(x=pos, color='white', linewidth=0.3, alpha=0.5)
plt.colorbar(im2, ax=ax2)

plt.suptitle("Head-to-Head Composition Scores", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Weight Processing for Cleaner Analysis

LayerNorm adds a confound to circuit analysis. The `fold_layer_norm` function absorbs LN's affine parameters into the adjacent weights, giving us cleaner circuit matrices.

In [ ]:
# Process model weights
processed_model = fold_layer_norm(model)
processed_model = center_unembed(processed_model)

# Verify outputs are preserved
logits_orig = model(tokens)
logits_folded = processed_model(tokens)
max_diff = float(jnp.max(jnp.abs(logits_orig - logits_folded)))
print(f"Max logit difference after folding: {max_diff:.6f}")
print("(Should be near zero - LN folding preserves the computation)")

# Now compare OV circuits before/after folding
ov_raw = ov_circuit(model, 9, 9)
ov_folded = ov_circuit(processed_model, 9, 9)

print(f"\nOV circuit norm (raw):    {float(ov_raw.norm()):.4f}")
print(f"OV circuit norm (folded): {float(ov_folded.norm()):.4f}")
print("(Folded norm includes the LN scaling, giving the actual magnitude)")

## 5. SVD Analysis of Head Circuits

The **singular values** of the OV circuit tell us the effective dimensionality of what a head can copy. A head with one dominant singular value is essentially a rank-1 "copy this direction" operation.

In [ ]:
# Analyze SVD of OV circuits for a few interesting heads
heads_to_analyze = [(9, 9), (9, 6), (10, 0), (0, 1)]

fig, axes = plt.subplots(1, len(heads_to_analyze), figsize=(4 * len(heads_to_analyze), 4))

for ax, (l, h) in zip(axes, heads_to_analyze):
    ov = ov_circuit(model, l, h)
    U, S, Vh = ov.svd()
    S_np = np.array(S)
    
    ax.bar(range(len(S_np)), S_np[:min(20, len(S_np))])
    ax.set_title(f"L{l}H{h}")
    ax.set_xlabel("Singular value index")
    ax.set_ylabel("Magnitude")
    ax.grid(True, alpha=0.3)

plt.suptitle("OV Circuit Singular Values (top 20)", fontsize=14)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **OV circuits** (`W_V @ W_O`) tell you what a head writes. The full OV circuit (`W_E @ W_V @ W_O @ W_U`) maps from input tokens to output logits.

2. **QK circuits** (`W_Q^T @ W_K`) tell you what a head attends to. They determine the attention pattern.

3. **Copy heads** have a positive diagonal in their full OV circuit — they promote the attended-to token.

4. **Direct logit attribution** projects each head's output through `W_U` to see its immediate effect on predictions.

5. **Composition scores** measure how strongly head A's output is used by head B, revealing multi-head circuits.

6. **Weight processing** (folding LN, centering) removes confounds for cleaner analysis.

### API Reference

```python
from irtk.circuits import (
    ov_circuit, qk_circuit,              # [d_model, d_model] FactoredMatrix
    full_ov_circuit, full_qk_circuit,    # [d_vocab, d_vocab] FactoredMatrix
    direct_logit_attribution,            # [n_layers, n_heads] per-head logit contribution
    residual_stream_attribution,         # component -> logit contribution dict
    all_composition_scores,              # [total_heads, total_heads] interaction matrix
)

from irtk.weight_processing import (
    fold_layer_norm,          # Absorb LN into weights
    center_writing_weights,   # Zero-mean writing weights
    center_unembed,           # Zero-mean unembedding
)
```